In [1]:
# code here
import cv2
import matplotlib.pyplot as plt
import numpy as np

ModuleNotFoundError: No module named 'cv2'

1)	Abrir as imagens coloridas e mostrar a imagem e seus histogramas (separados) cada canal de cor. Considerar os seguintes sistemas de cores:
a.	RGB
b.	HSV ou HSI
c.	Lab


In [ ]:
img1 = cv2.imread("image.jpg", cv2.IMREAD_COLOR)
img1RGB = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)

img2 = cv2.imread("image2.jpg", cv2.IMREAD_COLOR)
img2RGB = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

img3 = cv2.imread("Lenna.jpg", cv2.IMREAD_COLOR)
img3RGB = cv2.cvtColor(img3, cv2.COLOR_BGR2RGB)

plt.subplot(3,4,1)
plt.imshow(img1RGB);
plt.subplot(3,4,4*1+1)
plt.imshow(img2RGB);
plt.subplot(3,4,4*2+1)
plt.imshow(img3RGB);

def plot_color_spaces_and_histograms(image_path):
    # Carregar imagem e converter para os espaços de cor
    img_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Erro ao carregar a imagem: {image_path}")
        return
        
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

    # Dicionário com os espaços e seus nomes de canais
    spaces = {
        "RGB": (img_rgb, ['R', 'G', 'B'], ['red', 'green', 'blue']),
        "HSV": (img_hsv, ['H', 'S', 'V'], ['purple', 'cyan', 'black']),
        "LAB": (img_lab, ['L', 'a', 'b'], ['gray', 'magenta', 'yellow'])
    }

    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    fig.suptitle(f'Análise de Cores e Histogramas: {image_path}', fontsize=16)

    for i, (space_name, (img, channel_names, colors)) in enumerate(spaces.items()):
        # Plotar a imagem completa no primeiro quadro
        axes[i, 0].imshow(img if space_name == "RGB" else img_rgb)
        axes[i, 0].set_title(f"Imagem Original ({space_name})")
        axes[i, 0].axis('off')

        # Separar os canais
        channels = cv2.split(img)

        # Plotar o histograma para cada canal
        for j, (ch, name, color) in enumerate(zip(channels, channel_names, colors)):
            hist = cv2.calcHist([ch], [0], None, [256], [0, 256])
            axes[i, j+1].plot(hist, color=color)
            axes[i, j+1].set_title(f'Canal {name}')
            axes[i, j+1].set_xlim([0, 256])

    plt.tight_layout()
    plt.show()

# Testando com as três imagens (certifique-se de que estão na mesma pasta do script)
plot_color_spaces_and_histograms("image.jpg")
plot_color_spaces_and_histograms("image2.jpg")
plot_color_spaces_and_histograms("Lenna.jpg")


2)	Utilizando a imagem do mandrill, faça rotinas para detecção do focinho (região vermelha e azul). Utilize algum algoritmo de limiarização.

In [ ]:
img_mand = cv2.imread("mandrill.tiff", cv2.IMREAD_COLOR)
img_mand_RGB = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)
plt.imshow(img_mand_RGB);

# Carregar e converter
img_mand = cv2.imread("mandrill.tiff", cv2.IMREAD_COLOR)
img_mand_RGB = cv2.cvtColor(img_mand, cv2.COLOR_BGR2RGB)
img_mand_HSV = cv2.cvtColor(img_mand, cv2.COLOR_BGR2HSV)

# Definir os limites (limiares) para a cor AZUL no HSV
lower_blue = np.array([100, 50, 50])
upper_blue = np.array([140, 255, 255])
mask_blue = cv2.inRange(img_mand_HSV, lower_blue, upper_blue)

# Definir os limites para a cor VERMELHA no HSV
# O vermelho fica nas bordas do espectro HSV (0-10 e 170-180)
lower_red1 = np.array([0, 50, 50])
upper_red1 = np.array([10, 255, 255])
mask_red1 = cv2.inRange(img_mand_HSV, lower_red1, upper_red1)

lower_red2 = np.array([170, 50, 50])
upper_red2 = np.array([180, 255, 255])
mask_red2 = cv2.inRange(img_mand_HSV, lower_red2, upper_red2)

# Combinar as máscaras vermelhas
mask_red = cv2.bitwise_or(mask_red1, mask_red2)

# Combinar a máscara vermelha com a azul (Focinho Completo)
mask_snout = cv2.bitwise_or(mask_red, mask_blue)

# Aplicar a máscara final na imagem original RGB
snout_detected = cv2.bitwise_and(img_mand_RGB, img_mand_RGB, mask=mask_snout)

# Plotar os resultados
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1), plt.imshow(img_mand_RGB), plt.title("Original"), plt.axis('off')
plt.subplot(1, 3, 2), plt.imshow(mask_snout, cmap='gray'), plt.title("Máscara (Azul + Vermelho)"), plt.axis('off')
plt.subplot(1, 3, 3), plt.imshow(snout_detected), plt.title("Focinho Detectado"), plt.axis('off')
plt.show()


3)	Utilizando a imagem HE.jpg, tente detectar e contar a quantidade de núcleos celulares (em azul). Aplicar o Ostu para os 9 canais de cores e avaliar qual foi melhor (R,G,B,H,S,V,L,a,b)

In [ ]:
img_he = cv2.imread("HE.jpg", cv2.IMREAD_COLOR)
img_he_RGB = cv2.cvtColor(img_he, cv2.COLOR_BGR2RGB)
plt.imshow(img_he_RGB);

img_he = cv2.imread("HE.jpg", cv2.IMREAD_COLOR)
img_he_RGB = cv2.cvtColor(img_he, cv2.COLOR_BGR2RGB)
img_he_HSV = cv2.cvtColor(img_he, cv2.COLOR_BGR2HSV)
img_he_LAB = cv2.cvtColor(img_he, cv2.COLOR_BGR2LAB)

# Separar os 9 canais
R, G, B = cv2.split(img_he_RGB)
H, S, V = cv2.split(img_he_HSV)
L, a, b = cv2.split(img_he_LAB)

channels = {
    'R': R, 'G': G, 'B': B, 
    'H': H, 'S': S, 'V': V, 
    'L': L, 'a': a, 'b': b
}

plt.figure(figsize=(15, 12))
plt.suptitle('Limiarização de Otsu em 9 Canais (Detecção de Núcleos)', fontsize=16)

best_count = 0
best_channel = ""

for i, (name, ch) in enumerate(channels.items()):
    # Aplicar Filtro Gaussiano leve para reduzir ruído antes do Otsu
    blurred = cv2.GaussianBlur(ch, (5, 5), 0)
    
    # Aplicação do Threshold de Otsu
    # Nota: Dependendo do canal, o núcleo celular pode ser mais escuro ou mais claro. 
    # Em imagens H&E, o núcleo azul geralmente é escuro no canal Red e Luminescência,
    # então usamos THRESH_BINARY_INV. Caso a segmentação saia invertida (fundo branco),
    # o ideal é testar entre BINARY e BINARY_INV.
    ret, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Operação morfológica para limpar ruídos (abertura)
    kernel = np.ones((3,3), np.uint8)
    thresh_cleaned = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
    
    # Contar componentes (Núcleos)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(thresh_cleaned, connectivity=8)
    
    # Ignora o fundo (label 0)
    num_nuclei = num_labels - 1 
    
    plt.subplot(3, 3, i+1)
    plt.imshow(thresh_cleaned, cmap='gray')
    plt.title(f'Canal {name} | Núcleos: {num_nuclei}')
    plt.axis('off')

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

# Plotar a imagem original para comparação
plt.figure(figsize=(5, 5))
plt.imshow(img_he_RGB)
plt.title("Imagem HE Original")
plt.axis('off')
plt.show()

print("Nota Avaliativa: Após visualizar o plot, o canal que apresenta o melhor isolamento (fundo preto sólido e núcleos brancos bem definidos) geralmente é o canal 'a' do espaço LAB ou o canal 'S' do HSV para imagens do tipo H&E.")